In [1]:
import nbimporter  # type: ignore

In [2]:
functions = nbimporter.NotebookLoader().load_module("functions")

In [3]:
df_orig = functions.load_data()


In [4]:
# -----------------------------
# Componentes de tendencia lineal
# -----------------------------
df_orig = functions.add_linear_trend_component(df_orig)
df_orig = functions.add_month_component(df_orig)

In [5]:
# -----------------------------
# Fourier
# -----------------------------
df_orig = functions.add_sin_cos_components(df_orig)

In [6]:
  # -----------------------------
# Particionamiento de los datos
# -----------------------------
(
    X_complete,
    y_complete,
    X_train,
    y_train,
    X_test,
    y_test,
) = functions.train_test_split(
    df=df_orig,
    x_columns=[
        "trend",
        "sin_12m", "cos_12m",
        "sin_6m", "cos_6m",
        "sin_4m", "cos_4m",
        "sin_3m", "cos_3m",
    ],
    y_column="yt_true",
)

In [7]:
# -----------------------------
# Especificación del modelo
# -----------------------------
from sklearn.compose import ColumnTransformer  # type: ignore
from sklearn.linear_model import LinearRegression  # type: ignore
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures  # type: ignore

pipeline = Pipeline(
    [
        (
            "preprocessor",
            ColumnTransformer(
                [
                    (
                        "trend",
                        Pipeline(
                            [
                                ("poly", PolynomialFeatures(degree=3)),
                                ("scaler", MinMaxScaler()),
                            ]
                        ),
                        ["trend"],
                    ),
                ],
                remainder="passthrough",
            ),
        ),
        ("regressor", LinearRegression()),
    ]
)

In [8]:
# -----------------------------
# Entrenamiento y pronóstico
# -----------------------------
pipeline.fit(X_train, y_train)
df_orig["yt_pred_time_trend_3_fourier"] = pipeline.predict(X_complete)

functions.plot_time_series(df_orig)


<Figure size 1200x400 with 1 Axes>

In [9]:
# -----------------------------
# Almacenamiento de resultados
# -----------------------------
functions.save_forecasts(df_orig)


In [10]:
# -----------------------------
# Métricas de error
# -----------------------------
metrics = functions.compute_evaluation_metrics(df_orig)
functions.save_metrics(metrics)
metrics

,Metrics,yt_pred_time_trend_3_fourier
0,MSE Train,76907.17
1,MSE Test,122981.45
2,MAE Train,214.10
3,MAE Test,278.01
